# Task 1 - Word-level 1D CNN

Notebook ini menjalankan class `TextCNN` dari `src/word_level.py` menggunakan dataset IMDB.


In [ ]:
import re
from collections import Counter
from pathlib import Path

import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from src.word_level import TextCNN, HierarchicalTextCNN


## Download dan Baca Dataset

Bagian ini mengambil file CSV dari Kaggle, lalu membaca kolom `review` sebagai teks dan `sentiment` sebagai label.


In [ ]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

csv_path = next(Path(path).glob("*.csv"))
df = pd.read_csv(csv_path)

reviews = df["review"].tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()

print("Jumlah data:", len(reviews))
print("Contoh label:", labels[0])


## Tokenisasi, Vocabulary, dan Encoding

`TextCNN` tidak menerima teks mentah. Review harus diubah menjadi angka: tokenisasi memecah kalimat jadi kata, vocabulary memberi ID untuk setiap kata, encoding mengubah review menjadi deretan ID dengan panjang yang sama.


In [ ]:
def tokenize(text):
    return re.findall(r"[A-Za-z0-9']+", text.lower())


def encode_review(text, word_to_id, max_length):
    token_ids = [word_to_id.get(token, word_to_id["<unk>"]) for token in tokenize(text)]
    token_ids = token_ids[:max_length]
    token_ids += [word_to_id["<pad>"]] * (max_length - len(token_ids))
    return token_ids


In [ ]:
max_vocab_size = 50000
max_length = 200
batch_size = 3
num_epochs = 3

word_counter = Counter()
for review in reviews:
    word_counter.update(tokenize(review))

word_to_id = {"<pad>": 0, "<unk>": 1}

for word, _ in word_counter.most_common(max_vocab_size - 2):
    word_to_id[word] = len(word_to_id)

vocab_size = len(word_to_id)
print("Vocab size:", vocab_size)


## Buat DataLoader

`DataLoader` dipakai agar training loop bisa mengambil data per batch seperti contoh di PDF: `for batch_x, batch_y in train_dataloader`.


In [ ]:
encoded_reviews = [encode_review(review, word_to_id, max_length) for review in reviews]

batch_x = torch.tensor(encoded_reviews, dtype=torch.long)
batch_y = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(batch_x, batch_y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

print("Shape X:", batch_x.shape)
print("Shape y:", batch_y.shape)
print("Train data:", len(train_dataset))
print("Validation data:", len(val_dataset))


## Model, Loss, dan Optimizer

`TextCNN` mengeluarkan logits. Untuk training, `CrossEntropyLoss` memang menerima logits. Softmax dipakai saat prediksi, misalnya `torch.softmax(outputs, dim=1)`.


In [ ]:
model = TextCNN(
    vocab_size=vocab_size,
    embed_dim=300,
    num_filters=128,
    kernel_sizes=[2, 3, 4, 5],
    num_classes=2,
    pooling_type="max",
    sequence_length=max_length,
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


## Training Loop


In [ ]:
# Training loop example
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_dataloader:
        # batch_x: (batch, seq_len)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_dataloader):.4f}")


## Compare Pooling Performance

Bagian akhir ini langsung membandingkan `max`, `avg`, `none` atau without pooling, dan optional `adaptive`.


In [ ]:
def train_and_evaluate(pooling_type):
    model = TextCNN(
        vocab_size=vocab_size,
        embed_dim=300,
        num_filters=128,
        kernel_sizes=[2, 3, 4, 5],
        num_classes=2,
        pooling_type=pooling_type,
        sequence_length=max_length,
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        model.train()

        for batch_x, batch_y in train_dataloader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in val_dataloader:
            outputs = model(batch_x)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = probabilities.argmax(dim=1)
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)

    return correct / total


In [ ]:
for pooling_type in ["max", "avg", "none", "adaptive"]:
    accuracy = train_and_evaluate(pooling_type)
    print(f"Pooling: {pooling_type}, Accuracy: {accuracy:.4f}")


## Persiapan Task 2

Task 2 belum dikerjakan. Nanti bagian ini memakai dataset yang sama (`reviews` dan `labels`), tetapi inputnya diubah menjadi urutan karakter, bukan token kata.


In [ ]:
# TODO Task 2 nanti:
# 1. Buat character vocabulary, misalnya {"<pad>": 0, "<unk>": 1, "a": 2, ...}
# 2. Encode setiap review menjadi char ids dengan panjang tetap
# 3. Buat char_train_dataloader
# 4. Import CharCNN dari src.char_level
# 5. Train CharCNN dan bandingkan hasilnya dengan Word CNN


## Compare Word CNN vs Character CNN

Bagian ini diisi nanti setelah Task 2 selesai. Perbandingannya memakai dataset yang sama, terutama untuk teks slang, typo, atau rare words.


In [ ]:
# TODO setelah CharCNN selesai:
# print("Word CNN accuracy:", word_accuracy)
# print("Character CNN accuracy:", char_accuracy)


## Bonus Experiments

Bagian ini untuk bonus Word CNN: mencoba pooling yang berbeda dan model Hierarchical CNN. Jalankan nanti setelah eksperimen utama Word CNN stabil.


In [ ]:
# Bonus note:
# Adaptive pooling sudah ikut dibandingkan pada section Compare Pooling Performance.
# Untuk report, catat pooling mana yang accuracy-nya paling baik.


In [ ]:
# Bonus 2: Hierarchical CNN, yaitu beberapa Conv1D disusun bertingkat
hierarchical_model = HierarchicalTextCNN(
    vocab_size=vocab_size,
    embed_dim=300,
    num_filters=128,
    kernel_sizes=[3, 5, 7],
    num_classes=2,
    pooling_type="max",
)

hierarchical_model
